# UniProt GO Annotation Scraper

Fetches Molecular Function and Biological Process GO annotations from UniProt for a list of protein IDs.

Uses the [UniProt REST API](https://www.uniprot.org/help/api) — no browser driver required.

Based on work by Aksh77: https://github.com/Aksh77/Bio-Scraper/blob/master/UniProt-Scraper/scraper.py

In [ ]:
import csv
import time
import logging

import pandas as pd
import requests

logging.basicConfig(level=logging.WARNING, format='%(levelname)s: %(message)s')

In [ ]:
def display_list(arr):
    return ";\n".join(arr)


def fetch_uniprot_data(accession, retries=3, delay=2):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            return response.json()
        except Exception as e:
            logging.warning(f"Attempt {attempt + 1} failed for {accession}: {e}")
            if attempt < retries - 1:
                time.sleep(delay)
    logging.error(f"Max retries reached for {accession}")
    return None


def extract_go_terms(data, aspect_code):
    """Extract GO terms by aspect: 'F' = Molecular Function, 'P' = Biological Process, 'C' = Cellular Component."""
    terms = []
    if data is None:
        return terms
    prefix = aspect_code + ":"
    for ref in data.get("uniProtKBCrossReferences", []):
        if ref.get("database") == "GO":
            for prop in ref.get("properties", []):
                if prop.get("key") == "GoTerm":
                    value = prop.get("value", "")
                    if value.startswith(prefix):
                        terms.append(value[len(prefix):])
    return terms

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
INPUT_FILE  = "ProteinIDs_10min.csv"   # CSV with an 'IDs' column
OUTPUT_FILE = "Protein_data.csv"
RATE_LIMIT_DELAY = 0.3                 # seconds between API calls (be polite)
# ───────────────────────────────────────────────────────────────────────────────

df  = pd.read_csv(INPUT_FILE)
IDs = df["IDs"].tolist()
print(f"Loaded {len(IDs)} protein IDs from {INPUT_FILE}")

In [ ]:
with open(OUTPUT_FILE, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Protein ID", "Molecular Function-GO Annot", "Biological processes-GO Annot"])

    total = len(IDs)
    for idx, protein_id in enumerate(IDs):
        remaining = total - idx - 1
        print(f"[{idx + 1}/{total}] {protein_id}  ({remaining} remaining)")

        data = fetch_uniprot_data(str(protein_id))

        mol_functions = extract_go_terms(data, "F")
        bio_processes = extract_go_terms(data, "P")

        mol_str = display_list(mol_functions) if mol_functions else "No information"
        bio_str = display_list(bio_processes) if bio_processes else "No information"

        writer.writerow([protein_id, mol_str, bio_str])
        csvfile.flush()

        time.sleep(RATE_LIMIT_DELAY)

print(f"\nDone! Results saved to {OUTPUT_FILE}")